<h1 class="alert alert-block alert-info">Attention mechanism</h1>

In [1]:
import torch


In [2]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

<div class="alert alert-block alert-info">Dot product of Journey (which is second token) with all the tokens</div>

<img src="../Images/journey_token_context_vector.png" width="900">

<div>

In [3]:
query_second_token = inputs[1]
attn_score_for_second_token = torch.empty(inputs.shape[0])
for i, token_weight in enumerate(inputs):
    attn_score_for_second_token[i] = torch.dot(query_second_token, token_weight)

print(attn_score_for_second_token)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


<div class="alert alert-block alert-info">Normalize the attention weights of <span style="color:violet">Journey</span> using <span style="color:violet">Softmax</span></div>

<div class="alert alert-block alert-info">
E.g. attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])</br>
attention weights Sum: tensor(1.)
</div>

In [4]:
attn_weight_2_softmax = torch.softmax(attn_score_for_second_token, dim=0)
print(attn_weight_2_softmax)
print(attn_weight_2_softmax.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [5]:
context_vec_2 = torch.zeros(query_second_token.shape)
for i, dim_i in enumerate(inputs):
    context_vec_2_tmp = attn_weight_2_softmax[i] * dim_i
    context_vec_2 += context_vec_2_tmp
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [6]:
# There are 6 tokens
# attn_score = torch.empty(6,6)
# for i, row_i in enumerate(inputs):
#     for j, col_j in enumerate(inputs):
#         attn_score[i,j] = torch.dot(row_i, col_j)

# print(attn_score)

# matrix multiplication of Input and Input's transpose
# it short form and much more efficient than the above two for loop code
attn_score = inputs @ inputs.T

print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [7]:
attn_weights = torch.softmax(attn_score, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


<img src="../Images/attnscore_inputvector_multiplication.png" width="600">

In [8]:
context_vectors = attn_weights @ inputs
print(attn_weights.shape)
print(inputs.shape)
print(context_vectors)

torch.Size([6, 6])
torch.Size([6, 3])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


<h2 class="alert alert-block alert-info"> Attention with trainable weights</h2>

<img src="../Images/atten_with_trainable_weights.png" width=“500”>

In [9]:
import torch

# inputs = torch.tensor(
#   [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
#    [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
#    [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
# )
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# getting the columns
d_in = inputs.shape[1]

print(f"Total input dimension or context vector = {d_in}")
d_out = 2

print(inputs[0].shape)
print(inputs.shape[1])

# getting the rows
words = inputs.shape[0]
print(f"Total tokens/words = {words}")

print(f"Dimension out = {d_out}")



Total input dimension or context vector = 3
torch.Size([3])
3
Total tokens/words = 6
Dimension out = 2


In [10]:
#randomization
torch.manual_seed(123)

# Query, key and Value weight matrices
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print("keys.shape:", W_query.shape)
print("values.shape:", W_key.shape)
print("queries.shape:", W_value.shape)

keys.shape: torch.Size([3, 2])
values.shape: torch.Size([3, 2])
queries.shape: torch.Size([3, 2])


In [11]:
# multiply respective weight matrix with inputs
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)
print("queries.shape:", queries.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
queries.shape: torch.Size([6, 2])


In [12]:
# create attention score
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.1484, 0.2285, 0.2217, 0.1301, 0.0883, 0.1831],
        [0.1401, 0.2507, 0.2406, 0.1157, 0.0687, 0.1842],
        [0.1406, 0.2496, 0.2397, 0.1164, 0.0696, 0.1841],
        [0.1548, 0.2130, 0.2083, 0.1394, 0.1047, 0.1799],
        [0.1577, 0.2067, 0.2028, 0.1428, 0.1122, 0.1777],
        [0.1494, 0.2267, 0.2202, 0.1310, 0.0901, 0.1825]])


<h2 class="alert alert-block alert-info">Causal Attention with trainable weights. Mask right side of the words</h2>

<img src="../Images/atten_with_trainable_weights.png" width=“500”>

In [13]:
# apply mask on upper triangular
mask = torch.triu(torch.ones(words, words), diagonal=1)
print(mask)

# multiply with attn score
masked_attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked_attn_scores)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.9231,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.2705, 1.8524,   -inf,   -inf,   -inf,   -inf],
        [1.2544, 1.8284, 1.7877,   -inf,   -inf,   -inf],
        [0.6973, 1.0167, 0.9941, 0.5925,   -inf,   -inf],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707,   -inf],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])


In [14]:
masked_attn_scores = torch.softmax(masked_attn_scores, dim=-1)
print(masked_attn_scores)
masked_contextual_vectors =  masked_attn_scores @ values
print(masked_contextual_vectors)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3585, 0.6415, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2232, 0.3963, 0.3805, 0.0000, 0.0000, 0.0000],
        [0.2163, 0.2978, 0.2911, 0.1948, 0.0000, 0.0000],
        [0.1918, 0.2514, 0.2466, 0.1737, 0.1364, 0.0000],
        [0.1494, 0.2267, 0.2202, 0.1310, 0.0901, 0.1825]])
tensor([[0.1855, 0.8812],
        [0.3200, 0.9598],
        [0.3456, 0.9685],
        [0.3173, 0.8827],
        [0.2925, 0.8049],
        [0.3063, 0.8214]])
